# 1. Library Imports & Initialisation


In [1]:
import os
import json
import inspect
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, GRU, Dense
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Custom Application Modules
from sequence_store import SequenceStore
from predictor import FlowPredictor
from search import SearchCLI
from graph import Graph

# Global Constants for Reproducibility
RANDOM_SEED = 42
SEQ_LEN = 4
TEST_SPLIT = 0.2


# 2. Data Loading & Parsing


In [2]:
# Load raw SCATS data
df = pd.read_excel("Scats Data October 2006.xls", sheet_name="Data")

# Clean column names
df.columns = [str(col).strip() for col in df.columns]
df = df.rename(columns={
    "SCATS Number": "SCATS",
    "Start Time": "Date",
    "Unnamed: 0": "SCATS"
})

volume_cols = [col for col in df.columns if ":" in str(col)]
df = df[["SCATS", "Date"] + volume_cols]

df_long = df.melt(
    id_vars=["SCATS", "Date"],
    value_vars=volume_cols,
    var_name="time",
    value_name="flow"
)

df_long["flow"] = pd.to_numeric(df_long["flow"], errors="coerce")
df_long = df_long.dropna(subset=["flow"])

# Compute and save baseline average flow for Baseline Predictor
avg_flow_dict = df_long.groupby("SCATS")["flow"].mean().to_dict()
avg_flow_dict_str = {str(k): float(v) for k, v in avg_flow_dict.items()}
with open("baseline_avg_flow.json", "w") as f:
    json.dump(avg_flow_dict_str, f, indent=2)

df_long["Date"] = pd.to_datetime(df_long["Date"], errors="coerce")
df_long["time"] = df_long["time"].astype(str).str.strip()

df_long["timestamp"] = pd.to_datetime(
    df_long["Date"].dt.strftime("%Y-%m-%d") + " " + df_long["time"],
    errors="coerce"
)

df_long = df_long.dropna(subset=["SCATS", "timestamp", "flow"])
df_long["SCATS"] = df_long["SCATS"].astype(str).str.strip()

ts_df = df_long[["SCATS", "timestamp", "flow"]].copy()
ts_df = ts_df.groupby(["SCATS", "timestamp"], as_index=False)["flow"].sum()
ts_df = ts_df.sort_values(["SCATS", "timestamp"]).reset_index(drop=True)

print("SCATS dataset loaded:", len(ts_df), "records globally.")


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_18616\1608121278.py:34: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_long["timestamp"] = pd.to_datetime(


SCATS dataset loaded: 116160 records globally.


# 3. Time-Series Sequence Generation


In [3]:
def create_sequences_for_site(flow_values, seq_len=SEQ_LEN):
    X, y = [], []
    for i in range(len(flow_values) - seq_len):
        X.append(flow_values[i:i + seq_len])
        y.append(flow_values[i + seq_len])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X_list, y_list, site_list = [], [], []

for site_id, group in ts_df.groupby("SCATS"):
    group = group.sort_values("timestamp")
    flows = group["flow"].values.astype(np.float32)

    if len(flows) <= SEQ_LEN:
        continue

    X_site, y_site = create_sequences_for_site(flows, seq_len=SEQ_LEN)
    if len(X_site) == 0:
        continue

    X_list.append(X_site)
    y_list.append(y_site)
    site_list.extend([site_id] * len(X_site))

X = np.concatenate(X_list, axis=0)
y = np.concatenate(y_list, axis=0)
site_labels = np.array(site_list)

X = X.reshape((X.shape[0], X.shape[1], 1))

# Unroll the validation testing split
X_train, X_test, y_train, y_test, site_train, site_test = train_test_split(
    X, y, site_labels, test_size=TEST_SPLIT, random_state=RANDOM_SEED, shuffle=True
)

x_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

X_train_scaled = x_scaler.fit_transform(X_train.reshape(-1, 1)).reshape(X_train.shape)
X_test_scaled = x_scaler.transform(X_test.reshape(-1, 1)).reshape(X_test.shape)

y_train_scaled = y_scaler.fit_transform(y_train.reshape(-1, 1)).reshape(-1)
y_test_scaled = y_scaler.transform(y_test.reshape(-1, 1)).reshape(-1)



# 4. Deep Learning Architectures (LSTM & GRU)


In [4]:
# Build & Train LSTM Configuration
print("TRAINING LSTM MODEL:")
lstm_model = Sequential([
    LSTM(32, input_shape=(X_train_scaled.shape[1], 1)),
    Dense(1)
])
lstm_model.compile(optimizer='adam', loss='mse')
lstm_model.fit(X_train_scaled, y_train_scaled, epochs=5, batch_size=32, validation_split=0.1, verbose=1)
lstm_model.save("lstm_model.h5")

print("\nTRAINING GRU MODEL:")
# Build & Train GRU Configuration
gru_model = Sequential([
    GRU(32, input_shape=(X_train_scaled.shape[1], 1)),
    Dense(1)
])
gru_model.compile(optimizer='adam', loss='mse')
gru_model.fit(X_train_scaled, y_train_scaled, epochs=5, batch_size=32, validation_split=0.1, verbose=1)
gru_model.save("gru_model.h5")

# Export MinMax artifacts to backend graphs 
joblib.dump(x_scaler, "x_scaler.save")
joblib.dump(y_scaler, "y_scaler.save")


TRAINING LSTM MODEL:


c:\Users\LENOVO\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/5
2610/2610 ━━━━━━━━━━━━━━━━━━━━ 15s 5ms/step - loss: 0.0025 - val_loss: 0.0013
Epoch 2/5
2610/2610 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - loss: 0.0013 - val_loss: 0.0012
Epoch 3/5
2610/2610 ━━━━━━━━━━━━━━━━━━━━ 12s 5ms/step - loss: 0.0012 - val_loss: 0.0013
Epoch 4/5
2610/2610 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - loss: 0.0012 - val_loss: 0.0012
Epoch 5/5
2610/2610 ━━━━━━━━━━━━━━━━━━━━ 12s 5ms/step - loss: 0.0012 - val_loss: 0.0012



TRAINING GRU MODEL:
Epoch 1/5
2610/2610 ━━━━━━━━━━━━━━━━━━━━ 16s 5ms/step - loss: 0.0019 - val_loss: 0.0014
Epoch 2/5
2610/2610 ━━━━━━━━━━━━━━━━━━━━ 12s 5ms/step - loss: 0.0013 - val_loss: 0.0013
Epoch 3/5
2610/2610 ━━━━━━━━━━━━━━━━━━━━ 12s 5ms/step - loss: 0.0012 - val_loss: 0.0013
Epoch 4/5
2610/2610 ━━━━━━━━━━━━━━━━━━━━ 12s 5ms/step - loss: 0.0012 - val_loss: 0.0013
Epoch 5/5
2610/2610 ━━━━━━━━━━━━━━━━━━━━ 13s 5ms/step - loss: 0.0012 - val_loss: 0.0012


['y_scaler.save']

# 5. Machine Learning Architectures (Random Forest)


In [6]:
print('Extracting scaled variables across 2D boundaries for sklearn framework...')
X_test_rf = X_test_scaled.reshape(X_test_scaled.shape[0], -1)
X_train_rf = X_train_scaled.reshape(X_train_scaled.shape[0], -1)

print('Training Random Forest Regressor (This may take ~1 minute)...')
rf_model = RandomForestRegressor(n_estimators=100, max_depth=15, n_jobs=-1, random_state=RANDOM_SEED)
rf_model.fit(X_train_rf, y_train_scaled)

joblib.dump(rf_model, 'rf_model.pkl')
print('Training sequence completed! rf_model.pkl successfully exported.')


Extracting scaled variables across 2D boundaries for sklearn framework...
Training Random Forest Regressor (This may take ~1 minute)...
Training sequence completed! rf_model.pkl successfully exported.


# 6. Final Comprehensive Evaluation


In [7]:
print('Generating metric calculations over unseen test sequences...')

# Invert test-set predictions
y_pred_lstm_scaled = lstm_model.predict(X_test_scaled, verbose=0)
y_pred_lstm = y_scaler.inverse_transform(y_pred_lstm_scaled.reshape(-1, 1)).flatten()

y_pred_gru_scaled = gru_model.predict(X_test_scaled, verbose=0)
y_pred_gru = y_scaler.inverse_transform(y_pred_gru_scaled.reshape(-1, 1)).flatten()

y_pred_rf_scaled = rf_model.predict(X_test_rf)
y_pred_rf = y_scaler.inverse_transform(y_pred_rf_scaled.reshape(-1, 1)).flatten()

print('\n--- FINAL ARCHITECTURE EVALUATION METRICS ---')
models = [('LSTM', y_pred_lstm), ('GRU', y_pred_gru), ('Random Forest', y_pred_rf)]
print('-' * 40)
print(f"{'Model':<15} | {'MAE':>8} | {'RMSE':>8}")
print('-' * 40)
for name, y_pred in models:
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    print(f"{name:<15} | {mae:>8.2f} | {rmse:>8.2f}")
print('-' * 40)


Generating metric calculations over unseen test sequences...

--- FINAL ARCHITECTURE EVALUATION METRICS ---
----------------------------------------
Model           |      MAE |     RMSE
----------------------------------------
LSTM            |    31.88 |    49.11
GRU             |    32.00 |    49.22
Random Forest   |    30.40 |    47.14
----------------------------------------


In [3]:
import pandas as pd

# Don't guess the headers, just read the first 10 rows completely raw
df_raw = pd.read_csv("scats_mapping.csv", header=None, nrows=20)

print(df_raw.to_string())


                                                                                                                                                0                              1          2          3               4   5   6   7   8
0                                                                                                                             SCATS  Site Listing                            NaN        NaN        NaN             NaN NaN NaN NaN NaN
1                                     The list of Site descriptions and corresponding Site numbers below are used to request more detailed data.                             NaN        NaN        NaN             NaN NaN NaN NaN NaN
2   If multiple sites are required the format of the request must be a comma separated file containing the Site Number followed by the data range                            NaN        NaN        NaN             NaN NaN NaN NaN NaN
3                                                                           